# Diamond Price Prediction and Market Segmentation
## Phase 1 - Project Setup and Data Loading

This notebook implements the first project phase using the repository structure and config files defined in the project scaffold.

### Objectives
- Initialize a config-aware notebook workflow
- Load project YAML configuration files
- Resolve project paths dynamically
- Load the raw diamonds dataset
- Validate basic dataset structure
- Confirm target variable
- Inspect shape, columns, data types, duplicates, and descriptive statistics
- Create an initial column review artifact
- Write an initial project objective note for documentation

# Imports

In [2]:
from pathlib import Path
import pandas as pd

from src.utils.paths import (find_project_root, resolve_project_path, get_project_subdirs)
from src.utils.config import load_project_configs
from src.utils.io import (save_csv_file, save_json_file, save_text_file)

from src.data.load_data import load_raw_dataset
from src.data.validate_data import (check_target_present, get_missing_required_columns, get_unexpected_columns, count_duplicates, build_missing_summary, build_column_summary, build_dataset_summary)

In [3]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 1200)
pd.set_option("display.max_colwidth", 120)

## 1. Resolve project root

Find the repository root dynamically so the notebook works even when run from inside `notebooks/`.

In [4]:
project_root = find_project_root()
project_root

WindowsPath('F:/DATA SCIENCE/Projects/Diamond Dynamics Price Prediction and Market Segmentation')

## 2. Load project configuration files

Load all main YAML config files from the `configs/` directory.

In [5]:
configs = load_project_configs(project_root)

main_config = configs["main_config"]
paths_config = configs["paths_config"]
features_config = configs["features_config"]
model_registry = configs["model_registry"]
regression_config = configs["regression_config"]
clustering_config = configs["clustering_config"]
streamlit_config = configs["streamlit_config"]

print("Config files loaded successfully.")
print("Loaded configs:", list(configs.keys()))

Config files loaded successfully.
Loaded configs: ['main_config', 'paths_config', 'features_config', 'model_registry', 'regression_config', 'clustering_config', 'streamlit_config']


## 3. Read important settings from config

In [6]:
project_name = main_config["project"]["name"]
project_display_name = main_config["project"]["display_name"]
target_col = main_config["project"]["target_column"]
random_state = main_config["project"]["random_state"]

required_columns = features_config["schema"]["required_columns"]
numeric_features = features_config["schema"]["numeric_features"]
categorical_features = features_config["schema"]["categorical_features"]
expected_columns = features_config["schema"]["all_expected_features"]

print("Project:", project_display_name)
print("Target column:", target_col)
print("Random state:", random_state)

Project: Diamond Price Prediction and Market Segmentation
Target column: price
Random state: 42


## 4. Resolve important paths from `paths.yaml`

In [7]:
raw_data_file = resolve_project_path(project_root, paths_config["data"]["raw_data_file"])
external_reference_file = resolve_project_path(project_root, paths_config["data"]["external_reference_file"])

docs_dir = resolve_project_path(project_root, paths_config["docs"]["root"])
artifacts_dir = resolve_project_path(project_root, paths_config["artifacts"]["root"])
tables_dir = resolve_project_path(project_root, paths_config["artifacts"]["tables_dir"])
reports_dir = resolve_project_path(project_root, paths_config["artifacts"]["reports_dir"])

resolved_paths = {
    "RAW_DATA_FILE": raw_data_file,
    "EXTERNAL_REFERENCE_FILE": external_reference_file,
    "DOCS_DIR": docs_dir,
    "ARTIFACTS_DIR": artifacts_dir,
    "TABLES_DIR": tables_dir,
    "REPORTS_DIR": reports_dir,
}

resolved_paths

{'RAW_DATA_FILE': WindowsPath('F:/DATA SCIENCE/Projects/Diamond Dynamics Price Prediction and Market Segmentation/data/raw/diamonds.csv'),
 'EXTERNAL_REFERENCE_FILE': WindowsPath('F:/DATA SCIENCE/Projects/Diamond Dynamics Price Prediction and Market Segmentation/data/external/usd_inr_reference.csv'),
 'DOCS_DIR': WindowsPath('F:/DATA SCIENCE/Projects/Diamond Dynamics Price Prediction and Market Segmentation/docs'),
 'ARTIFACTS_DIR': WindowsPath('F:/DATA SCIENCE/Projects/Diamond Dynamics Price Prediction and Market Segmentation/artifacts'),
 'TABLES_DIR': WindowsPath('F:/DATA SCIENCE/Projects/Diamond Dynamics Price Prediction and Market Segmentation/artifacts/reports/tables'),
 'REPORTS_DIR': WindowsPath('F:/DATA SCIENCE/Projects/Diamond Dynamics Price Prediction and Market Segmentation/artifacts/reports')}

## 5. Project directory summary

In [8]:
get_project_subdirs(project_root)

{'project_root': WindowsPath('F:/DATA SCIENCE/Projects/Diamond Dynamics Price Prediction and Market Segmentation'),
 'data_dir': WindowsPath('F:/DATA SCIENCE/Projects/Diamond Dynamics Price Prediction and Market Segmentation/data'),
 'raw_dir': WindowsPath('F:/DATA SCIENCE/Projects/Diamond Dynamics Price Prediction and Market Segmentation/data/raw'),
 'interim_dir': WindowsPath('F:/DATA SCIENCE/Projects/Diamond Dynamics Price Prediction and Market Segmentation/data/interim'),
 'processed_dir': WindowsPath('F:/DATA SCIENCE/Projects/Diamond Dynamics Price Prediction and Market Segmentation/data/processed'),
 'external_dir': WindowsPath('F:/DATA SCIENCE/Projects/Diamond Dynamics Price Prediction and Market Segmentation/data/external'),
 'docs_dir': WindowsPath('F:/DATA SCIENCE/Projects/Diamond Dynamics Price Prediction and Market Segmentation/docs'),
 'notebooks_dir': WindowsPath('F:/DATA SCIENCE/Projects/Diamond Dynamics Price Prediction and Market Segmentation/notebooks'),
 'src_dir': W

## 6. Load raw dataset

In [10]:
df = load_raw_dataset(project_root)

print("Dataset loaded successfully.")
print("Raw file path:", raw_data_file)

Dataset loaded successfully.
Raw file path: F:\DATA SCIENCE\Projects\Diamond Dynamics Price Prediction and Market Segmentation\data\raw\diamonds.csv


## 7. Check dataset shape and columns

In [11]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (53940, 10)

Columns:
['carat', 'cut', 'color', 'clarity', 'depth', 'table', 'price', 'x', 'y', 'z']


## 8. Confirm target variable

In [12]:
target_present = check_target_present(df, target_col)
print("Target present:", target_present)

Target present: True


## 9. Preview rows

In [13]:
df.head()

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


## 10. Dataset structural information

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53940 entries, 0 to 53939
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   carat    53940 non-null  float64
 1   cut      53940 non-null  object 
 2   color    53940 non-null  object 
 3   clarity  53940 non-null  object 
 4   depth    53940 non-null  float64
 5   table    53940 non-null  float64
 6   price    53940 non-null  int64  
 7   x        53940 non-null  float64
 8   y        53940 non-null  float64
 9   z        53940 non-null  float64
dtypes: float64(6), int64(1), object(3)
memory usage: 4.1+ MB


## 11. Descriptive statistics

In [15]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
carat,53940.0,NaN,NaN,NaN,0.79794,0.474011,0.2,0.4,0.7,1.04,5.01
cut,53940,5,Ideal,21551,NaN,NaN,NaN,NaN,NaN,NaN,NaN
color,53940,7,G,11292,NaN,NaN,NaN,NaN,NaN,NaN,NaN
clarity,53940,8,SI1,13065,NaN,NaN,NaN,NaN,NaN,NaN,NaN
depth,53940.0,NaN,NaN,NaN,61.749405,1.432621,43.0,61.0,61.8,62.5,79.0
table,53940.0,NaN,NaN,NaN,57.457184,2.234491,43.0,56.0,57.0,59.0,95.0
price,53940.0,NaN,NaN,NaN,3932.799722,3989.439738,326.0,950.0,2401.0,5324.25,18823.0
x,53940.0,NaN,NaN,NaN,5.731157,1.121761,0.0,4.71,5.7,6.54,10.74
y,53940.0,NaN,NaN,NaN,5.734526,1.142135,0.0,4.72,5.71,6.54,58.9
z,53940.0,NaN,NaN,NaN,3.538734,0.705699,0.0,2.91,3.53,4.04,31.8


## 12. Data types

In [16]:
df.dtypes.to_frame(name="dtype")

,dtype
carat,float64
cut,object
color,object
clarity,object
depth,float64
table,float64
price,int64
x,float64
y,float64
z,float64


## 13. Duplicate row check

In [17]:
duplicate_count = count_duplicates(df)
print("Duplicate rows:", duplicate_count)

Duplicate rows: 146


## 14. Schema validation

In [18]:
missing_required_columns = get_missing_required_columns(df, required_columns)
unexpected_columns = get_unexpected_columns(df, expected_columns)

schema_validation_summary = {
    "target_present": target_present,
    "missing_required_columns": missing_required_columns,
    "unexpected_columns": unexpected_columns,
    "required_columns_count": len(required_columns),
    "actual_columns_count": len(df.columns),
}

schema_validation_summary

{'target_present': True,
 'missing_required_columns': [],
 'unexpected_columns': [],
 'required_columns_count': 10,
 'actual_columns_count': 10}

## 15. Missing value summary

In [19]:
missing_summary = build_missing_summary(df)
missing_summary

,column,missing_count,missing_pct
0,carat,0,0.0
1,clarity,0,0.0
2,color,0,0.0
3,cut,0,0.0
4,depth,0,0.0
5,price,0,0.0
6,table,0,0.0
7,x,0,0.0
8,y,0,0.0
9,z,0,0.0


## 16. Initial column review

In [20]:
column_summary = build_column_summary(df)
column_summary

,column,dtype,non null_count,null_count,null_pct,n_unique,sample_values
0,carat,float64,53940,0,0.0,273,"[0.23, 0.21, 0.23]"
1,cut,object,53940,0,0.0,5,"[Ideal, Premium, Good]"
2,color,object,53940,0,0.0,7,"[E, E, E]"
3,clarity,object,53940,0,0.0,8,"[SI2, SI1, VS1]"
4,depth,float64,53940,0,0.0,184,"[61.5, 59.8, 56.9]"
5,table,float64,53940,0,0.0,127,"[55.0, 61.0, 65.0]"
6,price,int64,53940,0,0.0,11602,"[326, 326, 327]"
7,x,float64,53940,0,0.0,554,"[3.95, 3.89, 4.05]"
8,y,float64,53940,0,0.0,552,"[3.98, 3.84, 4.07]"
9,z,float64,53940,0,0.0,375,"[2.43, 2.31, 2.31]"


## 17. Dataset summary dictionary

In [21]:
dataset_summary = build_dataset_summary(
    df=df,
    target_col=target_col,
    required_columns=required_columns,
    expected_columns=expected_columns,
)

dataset_summary

{'shape': {'rows': 53940, 'columns': 10},
 'columns': ['carat',
  'cut',
  'color',
  'clarity',
  'depth',
  'table',
  'price',
  'x',
  'y',
  'z'],
 'target_present': True,
 'missing_required_columns': [],
 'unexpected_columns': [],
 'duplicate_rows': 146,
 'dtypes': {'carat': 'float64',
  'cut': 'object',
  'color': 'object',
  'clarity': 'object',
  'depth': 'float64',
  'table': 'float64',
  'price': 'int64',
  'x': 'float64',
  'y': 'float64',
  'z': 'float64'}}

## 18. Save review artifacts

In [22]:
column_review_csv = tables_dir / "initial_column_review.csv"
missing_summary_csv = tables_dir / "initial_missing_value_summary.csv"
dataset_summary_json = reports_dir / "initial_dataset_summary.json"

save_csv_file(column_summary, column_review_csv, index=False)
save_csv_file(missing_summary, missing_summary_csv, index=False)
save_json_file(dataset_summary, dataset_summary_json)

print("Saved:")
print("-", column_review_csv)
print("-", missing_summary_csv)
print("-", dataset_summary_json)

Saved:
- F:\DATA SCIENCE\Projects\Diamond Dynamics Price Prediction and Market Segmentation\artifacts\reports\tables\initial_column_review.csv
- F:\DATA SCIENCE\Projects\Diamond Dynamics Price Prediction and Market Segmentation\artifacts\reports\tables\initial_missing_value_summary.csv
- F:\DATA SCIENCE\Projects\Diamond Dynamics Price Prediction and Market Segmentation\artifacts\reports\initial_dataset_summary.json


## 19. Initial project objective note

In [23]:
project_objective_note = f"""# Initial Project Objective Note

## Project
{project_display_name}

## Primary Goal
Build a machine learning system that predicts the `{target_col}` of a diamond
using its physical, dimensional, and quality-related attributes.

## Secondary Goal
Perform market segmentation to identify meaningful groups of diamonds based on
quality, dimensions, size-related properties, and pricing behavior.

## Phase 1 Focus
- set up the repository structure
- validate core configuration files
- load the raw dataset from the project path
- confirm schema and target variable
- inspect dataset structure, data types, duplicates, and descriptive statistics
- create the first documented data summary artifacts

## Expected Next Steps
- deeper data validation
- missing value and suspicious value checks
- preprocessing planning
- exploratory data analysis
- feature engineering
- regression modeling
- clustering and market segmentation
"""

print(project_objective_note)

# Initial Project Objective Note

## Project
Diamond Price Prediction and Market Segmentation

## Primary Goal
Build a machine learning system that predicts the `price` of a diamond
using its physical, dimensional, and quality-related attributes.

## Secondary Goal
Perform market segmentation to identify meaningful groups of diamonds based on
quality, dimensions, size-related properties, and pricing behavior.

## Phase 1 Focus
- set up the repository structure
- validate core configuration files
- load the raw dataset from the project path
- confirm schema and target variable
- inspect dataset structure, data types, duplicates, and descriptive statistics
- create the first documented data summary artifacts

## Expected Next Steps
- deeper data validation
- missing value and suspicious value checks
- preprocessing planning
- exploratory data analysis
- feature engineering
- regression modeling
- clustering and market segmentation



## 20. Save project objective note

In [24]:
objective_note_path = docs_dir / "initial_project_objective_note.md"
save_text_file(project_objective_note, objective_note_path)

print("Saved objective note to:", objective_note_path)

Saved objective note to: F:\DATA SCIENCE\Projects\Diamond Dynamics Price Prediction and Market Segmentation\docs\initial_project_objective_note.md
